# U APP — Kaggle Android Build Worker

Real Android SDK + Gradle build. The backend injects a `JOB` payload cell before pushing this notebook.

In [ ]:
import base64, json
JOB = None
SOURCE_B64 = ""


In [ ]:
import json, os, subprocess, sys, glob, shutil, zipfile, urllib.request, base64, time

WORK = "/kaggle/working"
INPUT = "/kaggle/input"
OUT = os.path.join(WORK, "out")
os.makedirs(OUT, exist_ok=True)
LOG_PATH = os.path.join(WORK, "build.log")
LOG = []

def log(line):
    text = str(line)
    LOG.append(text)
    print(text, flush=True)
    with open(LOG_PATH, "a") as fh:
        fh.write(text + "\n")

def run(cmd, cwd=None, check=True, env=None):
    log("$ " + (cmd if isinstance(cmd, str) else " ".join(cmd)))
    proc = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in (proc.stdout or "").splitlines()[-60:]:
        log(line)
    if check and proc.returncode != 0:
        raise RuntimeError("Command failed (%s): %s" % (proc.returncode, cmd))
    return proc.returncode

def find_job():
    job = globals().get("JOB")
    if isinstance(job, dict) and job.get("jobId"):
        log("Using the job payload embedded in this notebook.")
        return job, None
    for root, _, files in os.walk(INPUT):
        if "job.json" in files:
            with open(os.path.join(root, "job.json")) as fh:
                return json.load(fh), root
    raise RuntimeError("job.json missing from the attached Kaggle dataset")

def internet_ok():
    try:
        urllib.request.urlopen("https://dl.google.com/android/repository/repository2-1.xml", timeout=20).read(64)
        return True
    except Exception as exc:
        log("Internet check failed: %s" % exc)
        return False

def setup_toolchain():
    log("Installing JDK and build tools…")
    run("apt-get update -qq || sudo apt-get update -qq", check=False)
    run("(apt-get install -y -qq openjdk-17-jdk-headless unzip curl || sudo apt-get install -y -qq openjdk-17-jdk-headless unzip curl)", check=False)
    run("java -version", check=False)
    sdk = "/opt/android-sdk"
    if not os.path.isdir(os.path.join(sdk, "cmdline-tools", "latest")):
        os.makedirs(sdk, exist_ok=True)
        url = "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip"
        run(f"curl -sSL {url} -o /tmp/cmdline.zip && mkdir -p {sdk}/cmdline-tools && unzip -q -o /tmp/cmdline.zip -d {sdk}/cmdline-tools && mv {sdk}/cmdline-tools/cmdline-tools {sdk}/cmdline-tools/latest")
    env = dict(os.environ, ANDROID_SDK_ROOT=sdk, ANDROID_HOME=sdk)
    run(f"yes | {sdk}/cmdline-tools/latest/bin/sdkmanager --licenses", env=env, check=False)
    run(f"{sdk}/cmdline-tools/latest/bin/sdkmanager 'platform-tools' 'platforms;android-34' 'build-tools;34.0.0'", env=env)
    if not shutil.which("gradle"):
        run("curl -sSL https://services.gradle.org/distributions/gradle-8.9-bin.zip -o /tmp/gradle.zip && unzip -q -o /tmp/gradle.zip -d /opt && ln -sf /opt/gradle-8.9/bin/gradle /usr/local/bin/gradle")
    run("gradle --version", env=env, check=False)
    with open(os.path.join(os.getcwd(), "local.properties"), "w") as fh:
        fh.write("sdk.dir=%s\n" % sdk)
    return env, sdk

PERM_MAP = {"internet":"INTERNET","network":"ACCESS_NETWORK_STATE","notifications":"POST_NOTIFICATIONS","camera":"CAMERA","microphone":"RECORD_AUDIO","location":"ACCESS_FINE_LOCATION","bluetooth":"BLUETOOTH_CONNECT","nfc":"NFC","media":"READ_MEDIA_IMAGES","contacts":"READ_CONTACTS","calendar":"READ_CALENDAR","vibration":"VIBRATE","wakelock":"WAKE_LOCK","foreground":"FOREGROUND_SERVICE","storage":"READ_EXTERNAL_STORAGE","biometric":"USE_BIOMETRIC"}

def perm_name(value):
    key = str(value).lower()
    return PERM_MAP.get(key, str(value).upper())

def write_gradle(project, job):
    os.makedirs(os.path.join(project, "app", "src", "main", "java"), exist_ok=True)
    pkg = job["packageId"]
    pkg_path = pkg.replace(".", "/")
    java_dir = os.path.join(project, "app", "src", "main", "java", pkg_path)
    os.makedirs(java_dir, exist_ok=True)
    with open(os.path.join(project, "settings.gradle"), "w") as fh:
        fh.write("rootProject.name = '%s'\ninclude ':app'\n" % job["name"].replace("'", ""))
    with open(os.path.join(project, "build.gradle"), "w") as fh:
        fh.write("""buildscript {
    repositories { google(); mavenCentral() }
    dependencies { classpath 'com.android.tools.build:gradle:8.5.2' }
}
allprojects { repositories { google(); mavenCentral() } }
""")
    with open(os.path.join(project, "app", "build.gradle"), "w") as fh:
        fh.write("""apply plugin: 'com.android.application'
android {
    namespace '%s'
    compileSdk 34
    defaultConfig { applicationId '%s'; minSdk 24; targetSdk 34; versionCode 1; versionName '%s' }
    compileOptions { sourceCompatibility JavaVersion.VERSION_17; targetCompatibility JavaVersion.VERSION_17 }
}
""" % (pkg, pkg, job.get("version", "1.0.0")))
    manifest = """<?xml version="1.0" encoding="utf-8"?>
<manifest xmlns:android="http://schemas.android.com/apk/res/android">
%s
    <application android:label="%s" android:usesCleartextTraffic="false">
        <activity android:name=".MainActivity" android:exported="true">
            <intent-filter>
                <action android:name="android.intent.action.MAIN" />
                <category android:name="android.intent.category.LAUNCHER" />
            </intent-filter>
        </activity>
    </application>
</manifest>
""" % ("\n".join('    <uses-permission android:name="android.permission.%s" />' % perm_name(p) for p in (job.get("permissions") or ["internet", "network"])), job["name"])
    with open(os.path.join(project, "app", "src", "main", "AndroidManifest.xml"), "w") as fh:
        fh.write(manifest)
    src = job.get("source") or ""
    is_url = src.startswith("https://")
    with open(os.path.join(java_dir, "MainActivity.java"), "w") as fh:
        fh.write("""package %s;
import android.app.Activity;
import android.os.Bundle;
import android.webkit.WebView;
import android.webkit.WebViewClient;
public class MainActivity extends Activity {
  private WebView web;
  @Override protected void onCreate(Bundle state) {
    super.onCreate(state);
    web = new WebView(this);
    web.getSettings().setJavaScriptEnabled(true);
    web.getSettings().setDomStorageEnabled(true);
    web.getSettings().setAllowFileAccess(true);
    web.setWebViewClient(new WebViewClient());
    setContentView(web);
    %s
  }
  @Override public void onBackPressed() { if (web != null && web.canGoBack()) web.goBack(); else super.onBackPressed(); }
}
""" % (pkg, ('web.loadUrl("%s");' % src) if is_url else 'web.loadUrl("file:///android_asset/www/index.html");'))
    assets = os.path.join(project, "app", "src", "main", "assets", "www")
    os.makedirs(assets, exist_ok=True)
    with open(os.path.join(assets, "index.html"), "w") as fh:
        fh.write("""<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>%s</title><style>html,body{margin:0;height:100%%;background:%s}iframe{width:100%%;height:100%%;border:0;background:#fff}</style></head>
<body>%s</body></html>""" % (job["name"], job.get("splash") or "#080a0d",
        ('<iframe src="%s" allow="camera;microphone;geolocation;fullscreen"></iframe>' % src) if is_url else '<div style="color:#fff;font:600 16px system-ui;padding:24px">Add your HTML files to app/src/main/assets/www</div>'))

def main():
    job, input_root = find_job()
    app_root = os.path.join(WORK, "project")
    os.makedirs(app_root, exist_ok=True)
    log("U APP Kaggle worker start · job " + job["jobId"])
    log("App: %s · %s · %s" % (job["name"], job["packageId"], job.get("type")))
    if not internet_ok():
        raise RuntimeError("Kaggle notebook internet is disabled. Enable Internet in the notebook settings (phone verification required) so Android SDK dependencies can download.")
    env, sdk = setup_toolchain()
    uploaded_zip = None
    inline = globals().get("SOURCE_B64") or ""
    if inline:
        uploaded_zip = os.path.join(WORK, "source.zip")
        with open(uploaded_zip, "wb") as fh:
            fh.write(base64.b64decode(inline))
        log("Recovered the original source ZIP embedded in the build payload (%.2f MB)." % (os.path.getsize(uploaded_zip) / 1024 / 1024))
    if input_root:
        for root, _, files in os.walk(input_root):
            for name in files:
                if name.endswith(".zip") and not uploaded_zip:
                    uploaded_zip = os.path.join(root, name)
    project = app_root
    if uploaded_zip:
        log("Extracting preserved source: " + uploaded_zip)
        with zipfile.ZipFile(uploaded_zip) as z:
            for member in z.namelist():
                target = os.path.normpath(os.path.join(app_root, member))
                if not target.startswith(os.path.normpath(app_root) + os.sep) and target != os.path.normpath(app_root):
                    raise RuntimeError("Unsafe ZIP entry rejected: " + member)
            z.extractall(app_root)
        candidates = glob.glob(os.path.join(app_root, "**", "settings.gradle"), recursive=True)
        if candidates:
            project = os.path.dirname(candidates[0])
            log("Detected native Android project at " + project)
        else:
            write_gradle(app_root, job)
    else:
        write_gradle(app_root, job)
    with open(os.path.join(project, "local.properties"), "w") as fh:
        fh.write("sdk.dir=%s\n" % sdk)
    if os.path.exists(os.path.join(project, "gradlew")):
        os.chmod(os.path.join(project, "gradlew"), 0o755)
        build_cmd = ["./gradlew", "assembleDebug", "--no-daemon", "--stacktrace"]
    else:
        build_cmd = ["gradle", "assembleDebug", "--no-daemon", "--stacktrace", "-p", project]
    log("Building debug APK…")
    run(build_cmd, cwd=project, env=env)
    log("Attempting release AAB (unsigned)…")
    run((["gradle", "bundleRelease", "--no-daemon", "-p", project] if not os.path.exists(os.path.join(project, "gradlew")) else ["./gradlew", "bundleRelease", "--no-daemon"]), cwd=project, env=env, check=False)
    artifacts = glob.glob(os.path.join(project, "app", "build", "outputs", "**", "*.apk"), recursive=True) + glob.glob(os.path.join(project, "app", "build", "outputs", "**", "*.aab"), recursive=True)
    if not artifacts:
        raise RuntimeError("Gradle finished without producing an APK or AAB. See logs above.")
    for path in artifacts:
        dest = os.path.join(OUT, os.path.basename(path))
        shutil.copy2(path, dest)
        log("Artifact: %s (%.2f MB)" % (os.path.basename(path), os.path.getsize(path) / 1024 / 1024))
    shutil.copy2(LOG_PATH, os.path.join(OUT, "build.log"))
    if job.get("uploadUrl") and job.get("uploadToken"):
        boundary = "----uapp" + str(int(time.time()))
        for path in sorted(glob.glob(os.path.join(OUT, "*"))):
            try:
                with open(path, "rb") as fh:
                    body = b""
                    body += ("--%s\r\nContent-Disposition: form-data; name=\"file\"; filename=\"%s\"\r\nContent-Type: application/octet-stream\r\n\r\n" % (boundary, os.path.basename(path))).encode()
                    body += fh.read() + b"\r\n"
                    body += ("--%s\r\nContent-Disposition: form-data; name=\"jobId\"\r\n\r\n%s\r\n" % (boundary, job["jobId"])).encode()
                    body += ("--%s--\r\n" % boundary).encode()
                req = urllib.request.Request(job["uploadUrl"], data=body, method="POST", headers={
                    "Content-Type": "multipart/form-data; boundary=" + boundary,
                    "Authorization": "Bearer " + job["uploadToken"],
                })
                urllib.request.urlopen(req, timeout=120).read()
                log("Uploaded to Render: " + os.path.basename(path))
            except Exception as exc:
                log("Render upload failed for %s: %s (artifact remains in Kaggle output)" % (os.path.basename(path), exc))
    print("BUILD_RESULT " + json.dumps({"jobId": job["jobId"], "ok": True, "artifacts": [os.path.basename(p) for p in artifacts], "log": LOG[-80:]}))

if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        log("BUILD_ERROR: %s" % exc)
        print("BUILD_RESULT " + json.dumps({"ok": False, "error": str(exc), "log": LOG[-80:]}))
        sys.exit(1)
